# Gas Mixture Property Test

Functional tests for the CoolProp-based gas-mixture property layer.

The notebook checks:

- availability of gas-mixture APIs,
- mole-fraction / volume-fraction / mass-fraction composition handling,
- conversion of mass fractions to mole fractions,
- CoolProp mixture-string generation,
- runtime backend availability for `HEOS` and `REFPROP`,
- gas-phase imposition for user-defined gas mixtures,
- property retrieval for practical gas-mixture points,
- adapter compatibility with existing heat-transfer `FluidProps` containers:
  - `internal_flow.FluidProps`,
  - `internal_pressure_drop.FluidProps`,
  - `outside_flow.FluidProps`.

`REFPROP` is optional and user-provided. It is skipped if not installed,
licensed, and configured locally.

Core units:

- temperature: K
- pressure: Pa
- density: kg/m3
- dynamic viscosity: Pa*s
- thermal conductivity: W/(m*K)
- specific heat: J/(kg*K)

Composition basis:

- `mole`: mole fractions
- `volume`: volume fractions, treated as mole fractions for gases
- `mass`: mass fractions converted to mole fractions using molar masses

CoolProp gas-mixture calls use `imposed_phase="gas"` by default. This avoids
some automatic PT-flash failures for known gas-phase engineering states.

In [ ]:
from pathlib import Path
import sys
import math

import pandas as pd

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("Workspace root:", workspace_root)

In [ ]:
from core.properties import (
    GasMixturePropertyProvider,
    GasMixtureSpec,
    canonicalize_component_names,
    component_molar_mass,
    gas_mixture_full_props,
    gas_mixture_props,
    mass_fractions_to_mole_fractions,
)

from core.properties.coolprop_backend import (
    build_coolprop_mixture_string,
)

from core.properties.adapters import (
    to_internal_fluid_props,
    to_internal_pressure_drop_fluid_props,
    to_outside_fluid_props,
)

# Optional dependency check.
try:
    import CoolProp.CoolProp as CP
    COOLPROP_AVAILABLE = True
except ImportError:
    COOLPROP_AVAILABLE = False

assert component_molar_mass("N2") > 0.0
assert component_molar_mass("CO2") > 0.0
assert component_molar_mass("H2O") > 0.0

canonical = canonicalize_component_names(
    {
        "N2": 0.70,
        "O2": 0.05,
        "CO2": 0.20,
        "H2O": 0.05,
    }
)

assert "Nitrogen" in canonical
assert "Oxygen" in canonical
assert "CarbonDioxide" in canonical
assert "Water" in canonical

mole_from_mass = mass_fractions_to_mole_fractions(
    {
        "N2": 0.70,
        "O2": 0.05,
        "CO2": 0.20,
        "H2O": 0.05,
    }
)

assert abs(sum(mole_from_mass.values()) - 1.0) < 1e-12

mixture_string = build_coolprop_mixture_string(
    {
        "Nitrogen": 0.74,
        "Oxygen": 0.04,
        "CarbonDioxide": 0.12,
        "Water": 0.10,
    },
    backend="HEOS",
    normalize=True,
)

assert mixture_string.startswith("HEOS::")
assert "Nitrogen" in mixture_string
assert "CarbonDioxide" in mixture_string

print("Gas-mixture imports and API smoke test passed.")
print("CoolProp available:", COOLPROP_AVAILABLE)

if not COOLPROP_AVAILABLE:
    print("Install optional backend with: pip install -r requirements-coolprop.txt")
    print('or: pip install -e ".[coolprop]"')

In [ ]:
# Backend availability checks.
# HEOS should be available whenever CoolProp is installed.
# REFPROP is optional and should not make the test fail when unavailable.

import importlib
import core.properties.coolprop_backend as coolprop_backend

coolprop_backend = importlib.reload(coolprop_backend)

if hasattr(coolprop_backend, "check_coolprop_backend_availability"):
    check_backend = coolprop_backend.check_coolprop_backend_availability
else:
    # Fallback for older backend module versions loaded in kernel cache.
    def check_backend(backend: str):
        class BackendStatus:
            def __init__(self, backend: str, available: bool, message: str):
                self.backend = backend
                self.available = available
                self.message = message

            def __repr__(self) -> str:
                return f"BackendStatus(backend={self.backend!r}, available={self.available}, message={self.message!r})"

        try:
            import CoolProp.CoolProp as CP
        except ImportError:
            return BackendStatus(backend, False, "CoolProp is not installed.")

        test_fluid = "HEOS::Water" if backend.upper() == "HEOS" else f"{backend}::Water"

        try:
            CP.PropsSI("Dmass", "T", 300.0, "P", 101325.0, test_fluid)
            return BackendStatus(backend, True, f"CoolProp backend {backend!r} is available.")
        except Exception as exc:
            return BackendStatus(backend, False, f"CoolProp backend {backend!r} unavailable: {exc}")

heos_status = check_backend("HEOS")
refprop_status = check_backend("REFPROP")

print("HEOS:", heos_status)
print("REFPROP:", refprop_status)

assert heos_status.available == COOLPROP_AVAILABLE

if refprop_status.available:
    print("REFPROP is available locally and can be used for optional calculations.")
else:
    print("REFPROP is not available locally — optional REFPROP tests will be skipped.")

In [ ]:
gas_mixture_cases = [
    {
        "case": "air-like mole basis",
        "basis": "mole",
        "backend": "HEOS",
        "T_C": 20.0,
        "p_bar": 1.01325,
        "components": {
            "N2": 0.7808,
            "O2": 0.2095,
            "Ar": 0.0093,
            "CO2": 0.0004,
        },
    },
    {
        "case": "dry flue gas mole basis",
        "basis": "mole",
        "backend": "HEOS",
        "T_C": 180.0,
        "p_bar": 1.01325,
        "components": {
            "N2": 0.74,
            "O2": 0.04,
            "CO2": 0.12,
            "H2O": 0.10,
        },
    },
    {
        "case": "wet flue gas volume basis",
        "basis": "volume",
        "backend": "HEOS",
        "T_C": 180.0,
        "p_bar": 1.01325,
        "components": {
            "N2": 74.0,
            "O2": 4.0,
            "CO2": 12.0,
            "H2O": 10.0,
        },
    },
    {
        "case": "high CO2 wet gas mole basis",
        "basis": "mole",
        "backend": "HEOS",
        "T_C": 160.0,
        "p_bar": 1.01325,
        "components": {
            "N2": 0.62,
            "O2": 0.03,
            "CO2": 0.25,
            "H2O": 0.10,
        },
    },
    {
        "case": "mass basis flue gas",
        "basis": "mass",
        "backend": "HEOS",
        "T_C": 180.0,
        "p_bar": 1.01325,
        "components": {
            "N2": 0.70,
            "O2": 0.05,
            "CO2": 0.20,
            "H2O": 0.05,
        },
    },
    {
        "case": "hot dilute wet gas mass basis",
        "basis": "mass",
        "backend": "HEOS",
        "T_C": 250.0,
        "p_bar": 1.01325,
        "components": {
            "N2": 0.76,
            "O2": 0.06,
            "CO2": 0.15,
            "H2O": 0.03,
        },
    },
]

rows = []

for case in gas_mixture_cases:
    T = case["T_C"] + 273.15
    p = case["p_bar"] * 1.0e5

    try:
        spec = GasMixtureSpec(
            components=case["components"],
            basis=case["basis"],
            backend=case["backend"],
        )

        mole_fractions = spec.to_mole_fractions()
        fluid_string = build_coolprop_mixture_string(
            mole_fractions,
            backend=spec.backend,
            normalize=True,
        )

        props = None
        h_kJ_kg = math.nan
        phase = ""

        if COOLPROP_AVAILABLE:
            try:
                full = gas_mixture_full_props(
                    T=T,
                    p=p,
                    spec=spec,
                )
                props = full.transport
                h_kJ_kg = full.h / 1000.0
                phase = full.phase
                error = ""
            except Exception as exc:
                error = str(exc)
        else:
            error = "CoolProp not installed"

        rows.append(
            {
                "case": case["case"],
                "basis": case["basis"],
                "backend": case["backend"],
                "T_C": case["T_C"],
                "p_bar": case["p_bar"],
                "mole_fractions": mole_fractions,
                "fluid_string": fluid_string,
                "phase": phase,
                "rho_kg_m3": props.rho if props else math.nan,
                "mu_Pa_s": props.mu if props else math.nan,
                "k_W_mK": props.k if props else math.nan,
                "cp_J_kgK": props.cp if props else math.nan,
                "Pr": (props.mu * props.cp / props.k) if props else math.nan,
                "h_kJ_kg": h_kJ_kg,
                "error": error,
            }
        )

    except Exception as exc:
        rows.append(
            {
                "case": case["case"],
                "basis": case["basis"],
                "backend": case.get("backend", ""),
                "T_C": case["T_C"],
                "p_bar": case["p_bar"],
                "mole_fractions": {},
                "fluid_string": "",
                "phase": "",
                "rho_kg_m3": math.nan,
                "mu_Pa_s": math.nan,
                "k_W_mK": math.nan,
                "cp_J_kgK": math.nan,
                "Pr": math.nan,
                "h_kJ_kg": math.nan,
                "error": str(exc),
            }
        )

gas_mixture_df = pd.DataFrame(rows)

# API-level checks independent of CoolProp installation.
assert all(isinstance(row["mole_fractions"], dict) for _, row in gas_mixture_df.iterrows())
valid_api_rows = gas_mixture_df[
    (gas_mixture_df["mole_fractions"].apply(bool))
    & (gas_mixture_df["fluid_string"].astype(str) != "")
]
assert all(abs(sum(row["mole_fractions"].values()) - 1.0) < 1e-12 for _, row in valid_api_rows.iterrows())
assert all(
    str(row["fluid_string"]).startswith(f"{row['backend']}::")
    for _, row in valid_api_rows.iterrows()
)

if COOLPROP_AVAILABLE:
    # Mixture support depends on CoolProp backend data availability.
    # Validate all successful rows; keep readable errors for unsupported rows.
    assert len(gas_mixture_df) == len(gas_mixture_cases)

    successful = gas_mixture_df[gas_mixture_df["error"] == ""]
    if not successful.empty:
        assert (successful["rho_kg_m3"] > 0.0).all()
        assert (successful["mu_Pa_s"] > 0.0).all()
        assert (successful["k_W_mK"] > 0.0).all()
        assert (successful["cp_J_kgK"] > 0.0).all()
else:
    assert (gas_mixture_df["error"] == "CoolProp not installed").all()

print("Gas-mixture functional matrix test passed.")

gas_mixture_df

In [ ]:
# Demonstrate property-provider usage and adapter compatibility with existing
# heat-transfer FluidProps containers.

spec = GasMixtureSpec(
    components={
        "N2": 0.74,
        "O2": 0.04,
        "CO2": 0.12,
        "H2O": 0.10,
    },
    basis="mole",
    backend="HEOS",
)

provider = GasMixturePropertyProvider(spec)

if COOLPROP_AVAILABLE:
    try:
        props = provider.at(T=453.15, p=101325.0)

        internal_props = to_internal_fluid_props(props)
        internal_dp_props = to_internal_pressure_drop_fluid_props(props)
        outside_props = to_outside_fluid_props(props)

        assert internal_props.rho == props.rho
        assert internal_props.mu == props.mu
        assert internal_props.k == props.k
        assert internal_props.cp == props.cp

        assert internal_dp_props.rho == props.rho
        assert internal_dp_props.mu == props.mu
        assert internal_dp_props.k == props.k
        assert internal_dp_props.cp == props.cp

        assert outside_props.rho == props.rho
        assert outside_props.mu == props.mu
        assert outside_props.k == props.k
        assert outside_props.cp == props.cp

        print("Gas-mixture provider and heat-transfer adapter test passed.")
        print("internal_flow props:", internal_props)
        print("internal_pressure_drop props:", internal_dp_props)
        print("outside_flow props:", outside_props)
    except Exception as exc:
        print("Skipped provider numeric test because this mixture/state is unsupported by CoolProp.")
        print("Reason:", exc)
else:
    print("Skipped provider numeric test because CoolProp is not installed.")

In [ ]:
# Optional REFPROP calculation.
# This cell must not fail if REFPROP is not installed locally.

refprop_spec = GasMixtureSpec(
    components={
        "N2": 0.74,
        "O2": 0.04,
        "CO2": 0.12,
        "H2O": 0.10,
    },
    basis="mole",
    backend="REFPROP",
)

refprop_provider = GasMixturePropertyProvider(refprop_spec)
refprop_status = check_backend("REFPROP")

print(refprop_status)

if refprop_status.available:
    try:
        refprop_props = refprop_provider.at(T=453.15, p=101325.0)

        assert refprop_props.rho > 0.0
        assert refprop_props.mu > 0.0
        assert refprop_props.k > 0.0
        assert refprop_props.cp > 0.0

        print("REFPROP gas-mixture numeric test passed.")
        print(refprop_props)
    except Exception as exc:
        print("Skipped REFPROP numeric test because this mixture/state is unsupported by REFPROP/CoolProp.")
        print("Reason:", exc)
else:
    print("Skipped REFPROP numeric test because REFPROP is not available locally.")

In [ ]:
# Practical comparison: mole-basis and volume-basis should lead to the same mole fractions
# for equivalent gas compositions.

mole_spec = GasMixtureSpec(
    components={
        "N2": 0.74,
        "O2": 0.04,
        "CO2": 0.12,
        "H2O": 0.10,
    },
    basis="mole",
)

volume_spec = GasMixtureSpec(
    components={
        "N2": 74.0,
        "O2": 4.0,
        "CO2": 12.0,
        "H2O": 10.0,
    },
    basis="volume",
)

assert mole_spec.to_mole_fractions() == volume_spec.to_mole_fractions()

# Mass-basis conversion should give a normalized mole-fraction mapping,
# but not the same values as the original mass fractions.
mass_spec = GasMixtureSpec(
    components={
        "N2": 0.70,
        "O2": 0.05,
        "CO2": 0.20,
        "H2O": 0.05,
    },
    basis="mass",
)

mass_as_mole = mass_spec.to_mole_fractions()

assert abs(sum(mass_as_mole.values()) - 1.0) < 1e-12
assert mass_as_mole != mass_spec.normalized_components()

pd.DataFrame(
    [
        {
            "component": component,
            "mole_or_volume_fraction": mole_spec.to_mole_fractions().get(component, 0.0),
            "mass_basis_converted_to_mole_fraction": mass_as_mole.get(component, 0.0),
        }
        for component in sorted(set(mole_spec.to_mole_fractions()) | set(mass_as_mole))
    ]
)

In [ ]:
# Diagnostic: automatic PT flash vs imposed gas phase.
#
# This cell demonstrates the reason for gas-phase imposition in mixture calls.

if COOLPROP_AVAILABLE:
    from core.properties.coolprop_backend import CoolPropGasMixtureProvider

    components = {
        "Nitrogen": 0.70,
        "Oxygen": 0.20,
        "Water": 0.10,
    }

    auto_provider = CoolPropGasMixtureProvider(
        components=components,
        backend="HEOS",
        imposed_phase=None,
    )

    gas_provider = CoolPropGasMixtureProvider(
        components=components,
        backend="HEOS",
        imposed_phase="gas",
    )

    try:
        auto_props = auto_provider.at(T=433.15, p=101325.0)
        print("Automatic phase detection succeeded:", auto_props)
    except Exception as exc:
        print("Automatic phase detection failed, as expected for some CoolProp mixtures.")
        print("Reason:", exc)

    try:
        gas_props = gas_provider.at(T=433.15, p=101325.0)

        assert gas_props.rho > 0.0
        assert gas_props.mu > 0.0
        assert gas_props.k > 0.0
        assert gas_props.cp > 0.0

        print("Imposed gas-phase calculation succeeded:", gas_props)

    except Exception as exc:
        print("Imposed gas-phase calculation also failed for this CoolProp installation/state.")
        print("Reason:", exc)
else:
    print("Skipped diagnostic because CoolProp is not installed.")